In [ ]:
import pandas as pd
import re
import random
import string

#  CONFIGURATION 
INPUT_FILE = "a1_nutritional_literacy_qa.csv"
OUTPUT_FILE = "nutrients_task_qa_shuffled.csv"

def shuffle_multiple_choice(row):

    if row['type'] != 'multiple_choice':
        return row

    question_text = str(row['prompt'])
    original_answer_full = str(row['ground_truth']) 
    option_pattern = r'([A-Z])\)\s*(.*?)(?=\s*[A-Z]\)|$)'
    matches = re.findall(option_pattern, question_text, re.DOTALL)
    
    if not matches:
        return row

    # Extract the raw text of the options
    options_list = [m[1].strip() for m in matches]
    
    # Identify the correct answer text by stripping the leading "X) "
    # Example: "C) Grape leaves" -> "Grape leaves"
    correct_text = re.sub(r'^[A-Z]\)\s*', '', original_answer_full).strip()

    # Shuffle the options
    random.shuffle(options_list)

    # Dynamic Letter Mapping
    # ascii_uppercase provides A, B, C, D... dynamically based on count
    letters = list(string.ascii_uppercase[:len(options_list)])
    
    new_options_strings = []
    new_correct_letter = ""

    for i, opt_text in enumerate(options_list):
        current_letter = letters[i]
        new_options_strings.append(f"{current_letter}) {opt_text}")
        
        if opt_text == correct_text:
            new_correct_letter = current_letter

    # Extract the Question Stem (everything before "A)")
    # We use split instead of complex regex to ensure we don't lose the prompt
    stem = question_text.split('A)')[0].strip()
    
    # Rebuild the full prompt
    new_prompt = f"{stem}\n" + " ".join(new_options_strings)
    
    # Update row values
    row['prompt'] = new_prompt
    row['ground_truth'] = f"{new_correct_letter}) {correct_text}"
    
    return row

def main():
    try:
        # Load dataset with headers
        df = pd.read_csv(INPUT_FILE)
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    # Apply shuffling logic across the rows
    df = df.apply(shuffle_multiple_choice, axis=1)

    # Save output with headers
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Finished! Processed {len(df)} rows.")
    print(f"Shuffled dataset saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()